##Step 3: Calculate Customer Metrics

The dataset with which we are working consists of raw transactional history.  To apply the BTYD models, we need to derive several per-customer metrics:</p>

* **Frequency** - the number of dates on which a customer made a purchase subsequent to the date of the customer's first purchase
* **Age (Term)** - the number of time units, *e.g.* days, since the date of a customer's first purchase to the current date (or last date in the dataset)
* **Recency** - the age of the customer (as previously defined) at the time of their last purchase
* **Monetary Value** - the average per transaction-date spend by a customer during repeat purchases.  (Margin and other monetary values may also be used if available.)

It's important to note that when calculating metrics such as customer age that we need to consider when our dataset terminates.  Calculating these metrics relative to today's date can lead to erroneous results.  Given this, we will identify the last date in the dataset and define that as *today's date* for all calculations.

To get started with these calculations, let's take a look at how they are performed using the built-in functionality of the [btyd](https://btyd.readthedocs.io/en/latest/User%20Guide.html) library:

In [0]:
%pip install pymc-marketing==0.15.0 
dbutils.library.restartPython()

In [0]:
from pymc_marketing import clv
import pandas as pd
# import numpy as np
#from datetime import timedelta

# import btyd
# from btyd.fitters.beta_geo_fitter import BetaGeoFitter
# from btyd import GammaGammaFitter

# from btyd.plotting import plot_calibration_purchases_vs_holdout_purchases
# from btyd.plotting import plot_probability_alive_matrix
# from btyd.plotting import plot_frequency_recency_matrix


# import matplotlib.pyplot as plt

import pyspark.sql.functions as fn
from pyspark.sql.types import *

# import mlflow.pyfunc
# import mlflow

In [0]:
#read in dataset from last notebook and convert to pandas
orders = spark.read.format('delta').load('/tmp/clv/orders')
orders_pd = orders.toPandas()

# set the last transaction date as the end point for this historical dataset
current_date = orders_pd['InvoiceDate'].max()

# calculate the required customer metrics
# metrics_pd = (
#   clv.utils.summary_data_from_transaction_data(
#     orders_pd,
#     customer_id_col='CustomerID',
#     datetime_col='InvoiceDate',
#     observation_period_end = current_date, 
#     freq='D',
#     monetary_value_col='SalesAmount'  # use sales amount to determine monetary value
#     )
#   )

metrics_pd = clv.utils.rfm_summary(
    orders_pd,
    customer_id_col="CustomerID",
    datetime_col="InvoiceDate",
    monetary_value_col="SalesAmount", # use sales amount to determine monetary value
    datetime_format="%Y%m%d",
    time_unit="D",
)

# display first few rows
metrics_pd.head(10)

The btyd library, like many Python libraries, is single-threaded.  Using this library to derive customer metrics on larger transactional datasets may overwhelm your system or simply take too long to complete. For this reason, let's examine how these metrics can be calculated using the distributed capabilities of Apache Spark.

In the following cells we are going to use Programmatic Spark SQL API which may align better with some Data Scientist's preferences for complex data manipulation. Of course, you can derive the same results with Spark SQL using a SQL statement. In the code in the next cell, we first assemble each customer's order history consisting of the customer's ID, the date of their first purchase (first_at), the date on which a purchase was observed (transaction_at) and the current date (using the last date in the dataset for this value).  From this history, we can count the number of repeat transaction dates (frequency), the days between the last and first transaction dates (recency), the days between the current date and first transaction (T) and the associated monetary value (monetary_value) on a per-customer basis:


In [0]:
# programmatic sql api calls to derive summary customer stats
# valid customer orders
x = (
    orders
      .withColumn('transaction_at', fn.to_date('invoicedate'))
      .groupBy('customerid', 'transaction_at')
      .agg(fn.sum('salesamount').alias('salesamount'))   # SALES AMOUNT
    )

# calculate last date in dataset
y = (
  orders
    .groupBy()
    .agg(fn.max(fn.to_date('invoicedate')).alias('current_dt'))
  )

# calculate first transaction date by customer
z = (
  orders
    .groupBy('customerid')
    .agg(fn.min(fn.to_date('invoicedate')).alias('first_at'))
  )

# combine customer history with date info 
a = (x
    .crossJoin(y)
    .join(z, on='customerid', how='inner')
    .selectExpr(
      'customerid', 
      'first_at', 
      'transaction_at',
      'salesamount',
      'current_dt'
      )
    )

# calculate relevant metrics by customer
metrics_api = (a
           .groupBy(a.customerid, a.current_dt, a.first_at)
           .agg(
             (
              fn.countDistinct(a.transaction_at)-1).cast(FloatType()).alias('frequency'),
              fn.datediff(fn.max(a.transaction_at), a.first_at).cast(FloatType()).alias('recency'),
              fn.datediff(a.current_dt, a.first_at).cast(FloatType()).alias('T'),
              fn.when(fn.countDistinct(a.transaction_at)==1,0)                           # MONETARY VALUE
                .otherwise(
                  fn.sum(
                    fn.when(a.first_at==a.transaction_at,0)
                      .otherwise(a.salesamount)
                    )/(fn.countDistinct(a.transaction_at)-1)
                 ).alias('monetary_value')
               )
           .select('customerid','frequency','recency','T','monetary_value')
           .orderBy('customerid')
          )

display(metrics_api)

Let's take a moment to compare the data in these different metrics datasets, just to confirm the results are identical.  Instead of doing this record by record, let's calculate summary statistics across each dataset to verify their consistency:

NOTE You may notice means and standard deviations vary slightly in the hundred-thousandths and millionths decimal places.  This is a result of slight differences in data types between the pandas and Spark dataframes but do not affect our results in a meaningful way. 

In [0]:
# summary data from pymc-marketing
metrics_pd.describe()

In [0]:
# summary data from pyspark.sql API
metrics_api.toPandas().describe()

The metrics we've calculated represent summaries of a whole time series of data.  To support model validation and avoid overfitting, a common pattern with time series data is to train models on an earlier portion of the time series (known as the *calibration* period) and validate against a later portion of the time series (known as the *holdout* period). In the btyd library, the derivation of per customer metrics using calibration and holdout periods is done through a simple method call.  Because our dataset consists of a limited range for data, we will instruct this library method to use the last 90-days of data as the holdout period.  A simple parameter called a widget on the Databricks platform has been implemented to make the configuration of this setting easily changeable:

In [0]:
holdout_days = 90

In [0]:
# valid customer orders
x = (
  orders
    .withColumn('transaction_at', fn.to_date('invoicedate'))
    .groupBy('customerid', 'transaction_at')
    .agg(fn.sum('salesamount').alias('salesamount'))
  )

# calculate last date in dataset
y = (
  orders
    .groupBy()
    .agg(fn.max(fn.to_date('invoicedate')).alias('current_dt'))
  )

# calculate first transaction date by customer
z = (
  orders
    .groupBy('customerid')
    .agg(fn.min(fn.to_date('invoicedate')).alias('first_at'))
  )

# combine customer history with date info (CUSTOMER HISTORY)
p = (x
    .crossJoin(y)
    .join(z, on='customerid', how='inner')
    .withColumn('duration_holdout', fn.lit(holdout_days))
    .select(
      'customerid',
      'first_at',
      'transaction_at',
      'current_dt',
      'salesamount',
      'duration_holdout'
      )
     .distinct()
    ) 

# calculate relevant metrics by customer
# note: date_sub requires a single integer value unless employed within an expr() call
a = (p
       .where(p.transaction_at < fn.expr('date_sub(current_dt, duration_holdout)')) 
       .groupBy(p.customerid, p.current_dt, p.duration_holdout, p.first_at)
       .agg(
         (fn.countDistinct(p.transaction_at)-1).cast(FloatType()).alias('frequency_cal'),
         fn.datediff( fn.max(p.transaction_at), p.first_at).cast(FloatType()).alias('recency_cal'),
         fn.datediff( fn.expr('date_sub(current_dt, duration_holdout)'), p.first_at).cast(FloatType()).alias('T_cal'),
         fn.when(fn.countDistinct(p.transaction_at)==1,0)
           .otherwise(
             fn.sum(
               fn.when(p.first_at==p.transaction_at,0)
                 .otherwise(p.salesamount)
               )/(fn.countDistinct(p.transaction_at)-1)
             ).alias('monetary_value_cal')
       )
    )

b = (p
      .where((p.transaction_at >= fn.expr('date_sub(current_dt, duration_holdout)')) & (p.transaction_at <= p.current_dt) )
      .groupBy(p.customerid)
      .agg(
        fn.countDistinct(p.transaction_at).cast(FloatType()).alias('frequency_holdout'),
        fn.avg(p.salesamount).alias('monetary_value_holdout')
        )
   )

metrics_cal_api = (
                 a
                 .join(b, on='customerid', how='left')
                 .select(
                   'customerid',
                   'frequency_cal',
                   'recency_cal',
                   'T_cal',
                   'monetary_value_cal',
                   fn.coalesce(b.frequency_holdout, fn.lit(0.0)).alias('frequency_holdout'),
                   fn.coalesce(b.monetary_value_holdout, fn.lit(0.0)).alias('monetary_value_holdout'),
                   'duration_holdout'
                   )
                 .orderBy('customerid')
              )

display(metrics_cal_api)

Our data prep is nearly done.  The last thing we need to do is exclude customers for which we have no repeat purchases, *i.e.* frequency or frequency_cal is 0. The Pareto/NBD and BG/NBD models we will use focus exclusively on performing calculations on customers with repeat transactions.  A modified BG/NBD model, *i.e.* MBG/NBD, which allows for customers with no repeat transactions is supported by the btyd library.  However, to stick with the two most popular of the BYTD models in use today, we will limit our data to align with their requirements:

NOTE We are showing how both the pandas and Spark dataframes are filtered simply to be consistent with side-by-side comparisons earlier in this section of the notebook.  In a real-world implementation, you would simply choose to work with pandas or Spark dataframes for data preparation.

In [0]:
# remove customers with no repeats (complete dataset)
filtered_pd = metrics_pd[metrics_pd['frequency'] > 0]
filtered = metrics_api.where(metrics_api.frequency > 0)

## remove customers with no repeats in calibration period
filtered_cal = metrics_cal_api.where(metrics_cal_api.frequency_cal > 0)

Finally, we need to consider what to do about the negative daily totals found in our dataset.  Without any contextual information about the retailer from which this dataset is derived, we might assume these negative values represent returns.  Ideally, we'd match returns to their original purchases and adjust the monetary values for the original transaction date.  That said, we do not have the information required to consistently do this and so we will simply include the negative return values in our daily transaction totals. Where this causes a daily total to be £0 or lower, we will simply exclude that value from our analysis.  Outside of a demonstration setting, this would typically be inappropriate, but then again, you'd probably have access to the data required to properly reconcile these values:

In [0]:
# exclude dates with negative totals (see note above) 
filtered = filtered.where(filtered.monetary_value > 0)
filtered_cal = filtered_cal.where(filtered_cal.monetary_value_cal > 0)

Now that we've successfully preprocessed our data, it's time to train our model! In the next cell, we're going to persist the preprocessed dataframe for use in the next notebook.

In [0]:
filtered_cal.write.format('delta').mode('overwrite').option('overwriteSchema','true').save('/tmp/clv/preprocessed')